Project: /mediapipe/_project.yaml
Book: /mediapipe/_book.yaml

<link rel="stylesheet" href="/mediapipe/site.css">

# Hand gesture recognition model customization guide

<table align="left" class="buttons">
  <td>
    <a href="https://colab.research.google.com/github/googlesamples/mediapipe/blob/main/examples/customization/gesture_recognizer.ipynb" target="_blank">
      <img src="https://developers.google.com/static/mediapipe/solutions/customization/colab-logo-32px_1920.png" alt="Colab logo"> Run in Colab
    </a>
  </td>

  <td>
    <a href="https://github.com/googlesamples/mediapipe/blob/main/examples/customization/gesture_recognizer.ipynb" target="_blank">
      <img src="https://developers.google.com/static/mediapipe/solutions/customization/github-logo-32px_1920.png" alt="GitHub logo">
      View on GitHub
    </a>
  </td>
</table>

In [ ]:
#@title License information
# Copyright 2023 The MediaPipe Authors.
# Licensed under the Apache License, Version 2.0 (the "License");
#
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

O pacote MediaPipe Model Maker é uma solução de baixo código para personalizar modelos de Machine Learning (ML) em dispositivos.

Este notebook mostra o processo completo de personalização de um modelo de reconhecimento de gestos para reconhecer alguns gestos de mão comuns no dataset [HaGRID](https://www.kaggle.com/datasets/innominate817/hagrid-sample-30k-384p).

## Pré-requisitos

Instale o pacote MediaPipe Model Maker.

In [ ]:
!pip install --upgrade pip
!pip install mediapipe-model-maker

Project: /mediapipe/_project.yaml
Book: /mediapipe/_book.yaml

<link rel="stylesheet" href="/mediapipe/site.css">

# Guia de personalização do modelo de reconhecimento de gestos de mão

<table align="left" class="buttons">
  <td>
    <a href="https://colab.research.google.com/github/googlesamples/mediapipe/blob/main/examples/customization/gesture_recognizer.ipynb" target="_blank">
      <img src="https://developers.google.com/static/mediapipe/solutions/customization/colab-logo-32px_1920.png" alt="Colab logo"> Executar no Colab
    </a>
  </td>

  <td>
    <a href="https://github.com/googlesamples/mediapipe/blob/main/examples/customization/gesture_recognizer.ipynb" target="_blank">
      <img src="https://developers.google.com/static/mediapipe/solutions/customization/github-logo-32px_1920.png" alt="GitHub logo">
      Ver no GitHub
    </a>
  </td>
</table>

Importe as bibliotecas necessárias.

In [ ]:
from google.colab import files
import os
import shutil # Adicionado para operações de arquivo
import tensorflow as tf
assert tf.__version__.startswith('2')

from mediapipe_model_maker import gesture_recognizer

import matplotlib.pyplot as plt

## Exemplo Simples Completo

Este exemplo completo usa o Model Maker para personalizar um modelo para reconhecimento de gestos em dispositivos.

### Obter o dataset

O dataset para reconhecimento de gestos no Model Maker requer o seguinte formato: `<caminho_do_dataset>/<nome_do_rotulo>/<nome_da_imagem>.*`. Além disso, um dos nomes dos rótulos (`label_names`) deve ser `none`. O rótulo `none` representa qualquer gesto que não seja classificado como um dos outros gestos. Este exemplo agora usa o dataset 'libras' que foi processado e consolidado anteriormente.

In [ ]:
# 2. Descompacte o arquivo zip e reorganize os dados

# Substitua 'libras_dataset.zip' pelo nome do seu arquivo zip
!unzip libras.zip

# Assumindo que o arquivo zip cria uma pasta raiz como 'meu_libras_dataset'
# Ajuste conforme o nome da pasta raiz criada pelo seu zip
raw_dataset_root = "."
consolidated_dataset_path = "libras_consolidado"

os.makedirs(consolidated_dataset_path, exist_ok=True)

# Percorrer as pastas 'train' e 'test' (e outras se houver)
for split_folder in ['train', 'test']:
    split_path = os.path.join(raw_dataset_root, split_folder)
    if os.path.exists(split_path):
        for label_folder in os.listdir(split_path):
            label_path_src = os.path.join(split_path, label_folder)
            label_path_dst = os.path.join(consolidated_dataset_path, label_folder)

            if os.path.isdir(label_path_src):
                os.makedirs(label_path_dst, exist_ok=True)
                for filename in os.listdir(label_path_src):
                    src_file = os.path.join(label_path_src, filename)
                    dst_file = os.path.join(label_path_dst, filename)
                    # Evitar duplicatas se os nomes de arquivo forem os mesmos em train/test
                    if not os.path.exists(dst_file):
                        shutil.copy(src_file, dst_file)
                    else:
                        # Renomear se houver colisão de nomes, ou ignorar se for a mesma imagem
                        base, ext = os.path.splitext(filename)
                        count = 1
                        new_dst_file = os.path.join(label_path_dst, f"{base}_{count}{ext}")
                        while os.path.exists(new_dst_file):
                            count += 1
                            new_dst_file = os.path.join(label_path_dst, f"{base}_{count}{ext}")
                        shutil.copy(src_file, new_dst_file)

# 3. Defina o dataset_path para o novo diretório consolidado
dataset_path = consolidated_dataset_path

# Opcional: Crie uma pasta 'none' se ainda não existir e você quiser adicioná-la
# Isso é importante para o reconhecimento de gestos 'sem gesto'
none_label_path = os.path.join(dataset_path, 'none')
if not os.path.exists(none_label_path):
    os.makedirs(none_label_path)
    print(f"Pasta 'none' criada em: {none_label_path}. Por favor, adicione imagens de fundo ou sem gestos aqui se desejar.")

print(f"Dataset consolidado pronto em: {dataset_path}")

# 4. Verifique os rótulos do seu novo dataset consolidado
labels = []
for i in os.listdir(dataset_path):
  if os.path.isdir(os.path.join(dataset_path, i)):
    labels.append(i)
print(f"Rótulos detectados no dataset consolidado: {labels}")

Agora que seu dataset está preparado na estrutura correta (`libras_consolidado/<rótulo>/<imagem>.jpg`), você pode continuar com as etapas de carregamento e divisão do dataset, treinamento do modelo e avaliação, como demonstrado no notebook original.

In [ ]:
# Exemplo de como carregar os dados consolidados e dividir (como no notebook original)
# Certifique-se de que 'dataset_path' esteja definido corretamente para a pasta consolidada.

data = gesture_recognizer.Dataset.from_folder(
    dirname=dataset_path,
    hparams=gesture_recognizer.HandDataPreprocessingParams()
)
train_data, rest_data = data.split(0.8)
validation_data, test_data = rest_data.split(0.5)

print(f"Dados de treinamento: {len(train_data)} amostras")
print(f"Dados de validação: {len(validation_data)} amostras")
print(f"Dados de teste: {len(test_data)} amostras")


Verifique o dataset 'libras' imprimindo os rótulos. Lembre-se que um dos rótulos pode ser `none` se você o criou.

In [ ]:
print(dataset_path)
labels = []
for i in os.listdir(dataset_path):
  if os.path.isdir(os.path.join(dataset_path, i)):
    labels.append(i)
print(labels)

Para entender melhor o dataset, plote alguns exemplos de imagens para cada gesto.

In [ ]:
NUM_EXAMPLES = 5

for label in labels:
  label_dir = os.path.join(dataset_path, label)
  files_in_dir = [f for f in os.listdir(label_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

  if not files_in_dir:
    print(f"Aviso: Nenhuma imagem encontrada na pasta para o rótulo: {label}. Pulando visualização.")
    continue

  # Ajusta o número de exemplos se houver menos do que NUM_EXAMPLES
  n_to_show = min(len(files_in_dir), NUM_EXAMPLES)
  example_filenames = files_in_dir[:n_to_show]

  fig, axs = plt.subplots(1, n_to_show, figsize=(2 * n_to_show, 2))
  # Garante que axs seja uma lista mesmo se houver apenas 1 imagem
  if n_to_show == 1:
    axs = [axs]

  for i in range(n_to_show):
    img_path = os.path.join(label_dir, example_filenames[i])
    axs[i].imshow(plt.imread(img_path))
    axs[i].get_xaxis().set_visible(False)
    axs[i].get_yaxis().set_visible(False)

  fig.suptitle(f'Mostrando {n_to_show} exemplos para: {label}')
  plt.show()

### Executar o exemplo
O fluxo de trabalho consiste em 4 etapas que foram separadas em seus próprios blocos de código.

**Carregar o dataset**

Carregue o dataset localizado em `dataset_path` usando o método `Dataset.from_folder`. Ao carregar o dataset, execute o modelo pré-empacotado de detecção de mãos do MediaPipe Hands para detectar os marcos da mão a partir das imagens. Qualquer imagem sem mãos detectadas é omitida do dataset. O dataset resultante conterá as posições dos marcos da mão extraídas de cada imagem, em vez das próprias imagens.

A classe `HandDataPreprocessingParams` contém duas opções configuráveis para o processo de carregamento de dados:
* `shuffle`: Um booleano que controla se o dataset deve ser embaralhado. O padrão é `true`.
* `min_detection_confidence`: Um float entre 0 e 1 que controla o limite de confiança para detecção de mãos.

Divida o dataset: 80% para treinamento, 10% para validação e 10% para teste.

In [ ]:
data = gesture_recognizer.Dataset.from_folder(
    dirname=dataset_path,
    hparams=gesture_recognizer.HandDataPreprocessingParams()
)
train_data, rest_data = data.split(0.8)
validation_data, test_data = rest_data.split(0.5)

print(f"Dados de treinamento: {len(train_data)} amostras")
print(f"Dados de validação: {len(validation_data)} amostras")
print(f"Dados de teste: {len(test_data)} amostras")

**Treinar o modelo**

Treine o reconhecedor de gestos personalizado usando o método `create` e passando os dados de treinamento, dados de validação, opções do modelo e hiperparâmetros. Para obter mais informações sobre as opções do modelo e hiperparâmetros, consulte a seção [Hiperparâmetros](#hyperparameters) abaixo.

In [ ]:
hparams = gesture_recognizer.HParams(export_dir="exported_model")
options = gesture_recognizer.GestureRecognizerOptions(hparams=hparams)
model = gesture_recognizer.GestureRecognizer.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options
)

**Avaliar o desempenho do modelo**

Após treinar o modelo, avalie-o em um dataset de teste e imprima as métricas de perda e precisão.

In [ ]:
loss, acc = model.evaluate(test_data, batch_size=1)
print(f"Test loss:{loss}, Test accuracy:{acc}")

**Exportar para o Modelo Tensorflow Lite**

Após criar o modelo, converta-o e exporte-o para o formato de modelo Tensorflow Lite para uso posterior em uma aplicação em dispositivo. A exportação também inclui metadados do modelo, que incluem o arquivo de rótulos.

In [ ]:
model.export_model()
!ls exported_model

In [ ]:
files.download('exported_model/gesture_recognizer.task')

## Executar o modelo em dispositivo

Para usar o modelo TFLite em dispositivos através do MediaPipe Tasks, consulte a [página de visão geral](https://developers.google.com/mediapipe/solutions/vision/gesture_recognizer) do Gesture Recognizer.

## Hiperparâmetros {:#hyperparameters}

Você pode personalizar ainda mais o modelo usando a classe `GestureRecognizerOptions`, que possui dois parâmetros opcionais para `ModelOptions` e `HParams`. Use a classe `ModelOptions` para personalizar parâmetros relacionados ao próprio modelo e a classe `HParams` para personalizar outros parâmetros relacionados ao treinamento e salvamento do modelo.

`ModelOptions` possui um parâmetro personalizável que afeta a precisão:
* `dropout_rate`: A fração das unidades de entrada a serem descartadas. Usado na camada de dropout. O padrão é 0.05.
* `layer_widths`: Uma lista de larguras de camadas ocultas para o modelo de gesto. Cada elemento na lista criará uma nova camada oculta com a largura especificada. As camadas ocultas são separadas por BatchNorm, Dropout e ReLU. O padrão é uma lista vazia (sem camadas ocultas).

`HParams` possui a seguinte lista de parâmetros personalizáveis que afetam a precisão do modelo:
* `learning_rate`: A taxa de aprendizado a ser usada para o treinamento de descida de gradiente. O padrão é 0.001.
* `batch_size`: Tamanho do lote para treinamento. O padrão é 2.
* `epochs`: Número de iterações de treinamento sobre o dataset. O padrão é 10.
* `steps_per_epoch`: Um inteiro opcional que indica o número de etapas de treinamento por época. Se não for definido, o pipeline de treinamento calcula as etapas padrão por época como o tamanho do dataset de treinamento dividido pelo tamanho do lote.
* `shuffle`: Verdadeiro se o dataset for embaralhado antes do treinamento. O padrão é Falso.
* `lr_decay`: Decaimento da taxa de aprendizado a ser usada para o treinamento de descida de gradiente. O padrão é 0.99.
* `gamma`: Parâmetro Gamma para perda focal. O padrão é 2

Parâmetro `HParams` adicional que não afeta a precisão do modelo:
* `export_dir`: O local dos arquivos de checkpoint do modelo e dos arquivos de modelo exportados.

Por exemplo, o seguinte treina um novo modelo com uma taxa de dropout de 0.2 e taxa de aprendizado de 0.003.

In [ ]:
hparams = gesture_recognizer.HParams(learning_rate=0.003, export_dir="exported_model_2")
model_options = gesture_recognizer.ModelOptions(dropout_rate=0.2)
options = gesture_recognizer.GestureRecognizerOptions(model_options=model_options, hparams=hparams)
model_2 = gesture_recognizer.GestureRecognizer.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options
)

Avalie o modelo recém-treinado.

In [ ]:
loss, accuracy = model_2.evaluate(test_data)
print(f"Test loss:{loss}, Test accuracy:{accuracy}")

In [ ]:
model_2.export_model()
!ls exported_model_2
files.download('exported_model_2/gesture_recognizer.task')